# Section 5: Production Architecture & Cost Controls (20 Mins)

**Enterprise Use Case:** CloudCorp Smart Assistant (AskHR Agent)

**Notebook Goal:** Analyze enterprise production latency, implement Matryoshka Representation Learning (MRL) dimension reduction (768 -> 256), and audit lab consumption against our < .00 budget guardrail.


In [ ]:
# Step 4.0: Install Required Google Cloud Client Libraries
!pip install -q google-cloud-aiplatform>=1.60.0 numpy


In [1]:
# Step: Authenticate & Configure GCP Project (Colab OAuth or Local Service Account Key from .env)
import os
import time
import numpy as np

# 1. Load environment constants dynamically from .env file if available
def load_env_file():
    for path in [".env", "../.env", "cloudcorp-llms-on-gcp-lab/.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        val = v.strip().strip('"').strip("'")
                        if val:
                            os.environ.setdefault(k.strip(), val)
            print(f"Loaded configuration from '{path}'")
            return

load_env_file()

# 2. Project Configuration & Quota Project Settings
PROJECT_ID = os.environ.get("PROJECT_ID", "").strip() or os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or "your-gcp-project-id"  # @param {type:"string"}
REGION = os.environ.get("REGION", "us-central1").strip()               # @param {type:"string"}
BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-cloudcorp-policies").strip()

# Guardrail: Prevent running with unconfigured placeholder Project ID
if PROJECT_ID in ["your-gcp-project-id", "YOUR_PROJECT_ID", ""]:
    raise ValueError(
        "⚠️ ACTION REQUIRED: Please enter your Google Cloud Project ID in the PROJECT_ID field above! "
        "Ensure Vertex AI and Cloud Storage APIs are enabled."
    )

# Prevent 403 ADC Quota Project error (e.g. consumer projects/618104708054 in Cloud Shell)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

# 3. Authentication Option A: Google Colab OAuth
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated via Google Colab OAuth.")
except ImportError:
    print("ℹ️ Running outside Google Colab (Cloud Shell / Local Machine / IDE).")

# 4. Authentication Option B: Service Account JSON Key via .env / Local File
sa_key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "")
if sa_key_path:
    candidate_key_paths = [
        sa_key_path,
        os.path.join("..", sa_key_path),
        os.path.join("cloudcorp-llms-on-gcp-lab", sa_key_path),
        os.path.expanduser(f"~/Documents/{sa_key_path}"),
        os.path.expanduser(f"~/Downloads/{sa_key_path}")
    ]
    resolved_key = next((p for p in candidate_key_paths if os.path.isfile(p)), None)
    if resolved_key:
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(resolved_key)
        print(f"🔑 Loaded Service Account Key: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
    else:
        print(f"⚠️ Key file '{sa_key_path}' specified in .env not found on disk.")

# 5. Verify Active Credentials & Bind Quota Project
import google.auth
credentials = None
try:
    credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
    if hasattr(credentials, "with_quota_project"):
        credentials = credentials.with_quota_project(PROJECT_ID)
    account_id = getattr(credentials, "service_account_email", None) or getattr(credentials, "signer_email", "Google User Account (OAuth/ADC)")
    print(f"✅ Active Identity: {account_id}")
    print(f"✅ Quota Project bound: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️ Warning: No active credentials detected ({e})")
    print("   • In Colab: Run auth.authenticate_user()")
    print("   • In Cloud Shell: Run 'gcloud auth application-default set-quota-project '")
    print("   • Locally: Set GOOGLE_APPLICATION_CREDENTIALS in .env or run 'gcloud auth application-default login'")

# 6. Initialize Vertex AI with explicit credentials
import vertexai
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel

vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
embed_model = TextEmbeddingModel.from_pretrained("text-embedding-004")
print(f"✅ Vertex AI Embeddings initialized for project '{PROJECT_ID}' in '{REGION}'.")

vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
flash_model = GenerativeModel("gemini-3.5-flash-lite")
print(f"✅ Vertex AI GenerativeModel (gemini-3.5-flash-lite) loaded successfully.")


Loaded configuration from '../.env'
ℹ️ Running outside Google Colab (Local Machine / IDE).
🔑 Loaded Service Account Key: /Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/notebooks/gcp-key.json
✅ Active Identity: cloudcorp-lab-sa@upgradlabs-1751346914784.iam.gserviceaccount.com


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


✅ Vertex AI Embeddings initialized for project 'upgradlabs-1751346914784' in 'us-central1'.
✅ Vertex AI GenerativeModel (gemini-3.5-flash-lite) loaded successfully.


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [2]:
# Step 4.1: Latency Benchmarking (RAG Retrieval vs Multi-Turn Agent)
import time

vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
t0 = time.time()
_ = embed_model.get_embeddings(["Sample policy query"])
t_embed = (time.time() - t0) * 1000

vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
t0 = time.time()
_ = flash_model.generate_content("Explain PTO carryover rules in one sentence.")
t_llm = (time.time() - t0) * 1000

print("=== Latency Benchmark Profiles ===")
print(f"• text-embedding-004 latency   : {t_embed:.1f} ms")
print(f"• gemini-3.5-flash-lite latency: {t_llm:.1f} ms")
print(f"• Pure RAG Total Turnaround    : ~{(t_embed + t_llm):.1f} ms")
print(f"• Agentic Loop (2 LLM Turns)   : ~{(t_embed + (t_llm * 2) + 50):.1f} ms")


=== Latency Benchmark Profiles ===
• text-embedding-004 latency   : 1775.0 ms
• gemini-3.5-flash-lite latency: 1222.2 ms
• Pure RAG Total Turnaround    : ~2997.2 ms
• Agentic Loop (2 LLM Turns)   : ~4269.3 ms


In [ ]:
# Step 4.2: Matryoshka Dimension Reduction (768 -> 256 dimensions)
# text-embedding-004 supports Matryoshka Representation Learning (MRL)
full_emb = embed_model.get_embeddings(["CloudCorp hybrid work stipend reimbursement"])[0]
full_vec = np.array(full_emb.values)

# Truncate to first 256 dimensions and re-normalize to unit length
mrl_256_vec = full_vec[:256]
mrl_256_vec = mrl_256_vec / np.linalg.norm(mrl_256_vec)

print(f"Full Vector Dimensions   : {len(full_vec)} floats ({len(full_vec) * 4} bytes)")
print(f"MRL Vector Dimensions    : {len(mrl_256_vec)} floats ({len(mrl_256_vec) * 4} bytes)")
print(f"Memory / Index Savings   : {(1 - len(mrl_256_vec) / len(full_vec)) * 100:.1f}%")


In [3]:
# Step 4.3: Workshop Cost Auditor (< $5.00 Budget Verification)
def calculate_lab_cost(num_queries=30, num_chunks=25):
    # GCP Pricing (us-central1)
    COST_EMBED_PER_1K_CHARS = 0.000025
    COST_GEMINI_INPUT_PER_1M = 0.075
    COST_GEMINI_OUTPUT_PER_1M = 0.30
    
    # Lab consumption
    total_embed_chars = num_chunks * 600 + (num_queries * 60)
    embed_cost = (total_embed_chars / 1000) * COST_EMBED_PER_1K_CHARS
    
    input_tokens = num_queries * 800
    output_tokens = num_queries * 150
    llm_cost = (input_tokens / 1_000_000 * COST_GEMINI_INPUT_PER_1M) + (output_tokens / 1_000_000 * COST_GEMINI_OUTPUT_PER_1M)
    
    gcs_cost = 0.0005  # < 1 MB data transfer & storage
    cloud_run_cost = 0.0  # Within free tier (2M requests/mo)
    cloud_func_cost = 0.0 # Within free tier (2M requests/mo)
    
    total = embed_cost + llm_cost + gcs_cost + cloud_run_cost + cloud_func_cost
    
    print("================ LAB COST BREAKDOWN ================")
    print(f"• Vertex AI Embeddings (text-embedding-004) : ${embed_cost:.5f}")
    print(f"• Vertex AI Gemini (gemini-3.5-flash-lite)       : ${llm_cost:.5f}")
    print(f"• Google Cloud Storage                      : ${gcs_cost:.5f}")
    print(f"• Cloud Run & Cloud Functions Free Tier     : $0.00000")
    print(f"----------------------------------------------------")
    print(f"• TOTAL ESTIMATED LAB SPEND                 : ${total:.4f}")
    print(f"• BUDGET GUARDRAIL CAP                      : $5.0000")
    print(f"• STATUS                                    : {'✅ UNDER BUDGET' if total < 5.0 else '❌ OVER BUDGET'}")
    print("====================================================")

calculate_lab_cost()


================ LAB COST BREAKDOWN ================
• Vertex AI Embeddings (text-embedding-004) : $0.00042
• Vertex AI Gemini (gemini-3.5-flash-lite)       : $0.00315
• Google Cloud Storage                      : $0.00050
• Cloud Run & Cloud Functions Free Tier     : $0.00000
----------------------------------------------------
• TOTAL ESTIMATED LAB SPEND                 : $0.0041
• BUDGET GUARDRAIL CAP                      : $5.0000
• STATUS                                    : ✅ UNDER BUDGET
